In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [3]:
# loading dataset
df = pd.read_csv(r'C:\DS-Projects\iv-surface-prediction\data\processed\dataset_long_pruned.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
dates = sorted(df['datetime'].dt.date.unique())

EXPIRY_DATE = dates[-1]  # Jan 27

FOLDS = [
    {'name': 'Fold 1', 'train': dates[:9],  'val': dates[9:11]},
    {'name': 'Fold 2', 'train': dates[:10], 'val': dates[10:12]},
]
FEATURE_COLS = [
    'iv_neighbor_mean',
    'iv_roll_mean_5',
    'iv_roll_mean_10',
    'wide_iv_neighbor_mean',
    'moneyness',
    'iv_neighbor_+1',
    'mean_pe_iv',
    'iv_neighbor_+2',
    'dist_from_atm_pct',
    'mean_ce_iv',
    'strike',
    'dist_from_atm',
    'iv_neighbor_-1',
    'iv_neighbor_-2',
    'strike_rank',
    'log_moneyness'
]

In [4]:
class IVSurfaceMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.1):
        super().__init__()
        layers = []; prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# FIX 1: train_mlp now returns model, scaler, best_epoch, train_losses, val_losses (5 values)
def train_mlp(X_train, y_train, X_val, y_val, epochs=500, batch_size=256, patience=40):

    scaler = StandardScaler()
    X_train_trf = scaler.fit_transform(X_train)
    X_val_trf   = scaler.transform(X_val)

    Xtt = torch.tensor(X_train_trf, dtype=torch.float32)
    ytt = torch.tensor(y_train,     dtype=torch.float32)
    Xvt = torch.tensor(X_val_trf,   dtype=torch.float32)
    yvt = torch.tensor(y_val,       dtype=torch.float32)

    loader  = DataLoader(TensorDataset(Xtt, ytt), batch_size=batch_size, shuffle=True)
    model   = IVSurfaceMLP(X_train.shape[1])
    opt     = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()

    best_loss  = float('inf')
    best_state = None
    best_epoch = 0
    no_imp     = 0
    train_losses = []
    val_losses   = []

    for epoch in range(epochs):
        model.train()
        epoch_train_loss = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_train_loss += loss.item()
        sched.step()

        train_losses.append(epoch_train_loss / len(loader))

        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xvt), yvt).item()
        val_losses.append(vl)

        if vl < best_loss:
            best_loss  = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            no_imp     = 0
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    model.load_state_dict(best_state)
    return model, scaler, best_epoch, train_losses, val_losses


def predict_mlp(model, scaler, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(scaler.transform(X), dtype=torch.float32)).numpy()


def predict_interp(mask_rows, df_context):
    preds = []
    for _, row in mask_rows.iterrows():
        ts_data = df_context[
            (df_context['datetime']   == row['datetime']) &
            (df_context['option_type'] == row['option_type']) &
            (df_context['iv'].notna())
        ].sort_values('strike')
        if len(ts_data) >= 3:
            try:
                f = interp1d(
                    ts_data['strike'], ts_data['iv'],
                    kind='linear', bounds_error=False, fill_value='extrapolate'
                )
                preds.append(float(f(row['strike'])))
            except:
                preds.append(np.nan)
        else:
            preds.append(np.nan)
    return np.array(preds)


def make_holdout(val_df):
    np.random.seed(42)
    available_rows = val_df[val_df['iv'].notna()].index
    hidden_rows    = np.random.choice(available_rows, size=int(0.2 * len(available_rows)), replace=False)
    ground_truth   = val_df.loc[hidden_rows, 'iv'].values
    val_masked     = val_df.copy()
    val_masked.loc[hidden_rows, 'iv'] = np.nan
    return val_masked, hidden_rows, ground_truth


def predict_cascaded_full(missing_rows, df_context, mlp_model, mlp_scaler, expiry_date):
    
    preds = np.full(len(missing_rows), np.nan)
    
    expiry_mask = missing_rows['datetime'].dt.date == expiry_date

    # --- Expiry day: interpolation ---
    if expiry_mask.any():
        expiry_rows = missing_rows[expiry_mask]
        preds[expiry_mask.values] = predict_interp(expiry_rows, df_context)

    # --- Non-expiry days: MLP ---
    non_expiry_mask = ~expiry_mask
    if non_expiry_mask.any():
        non_expiry_rows = missing_rows[non_expiry_mask]
        X = non_expiry_rows[FEATURE_COLS].fillna(0).values
        preds[non_expiry_mask.values] = predict_mlp(mlp_model, mlp_scaler, X)

    return preds

In [5]:
all_fold_preds = []

for fold in FOLDS:

    train_df = df[df['datetime'].dt.date.isin(fold['train'])].copy()
    val_df   = df[df['datetime'].dt.date.isin(fold['val'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)
    mask_rows = val_masked.loc[hidden_rows]

    train_obs = train_df[train_df['iv'].notna()]
    X_all     = train_obs[FEATURE_COLS].fillna(0).values
    y_all     = train_obs['iv'].values

    split = int(0.85 * len(X_all))
    model, scaler, best_ep, train_losses, val_losses = train_mlp(
        X_all[:split], y_all[:split],
        X_all[split:], y_all[split:]
    )

    p_interp = predict_interp(mask_rows, val_masked)
    p_mlp    = predict_mlp(model, scaler, mask_rows[FEATURE_COLS].fillna(0).values)

    all_fold_preds.append({
        'gt':       ground_truth,
        'p_interp': p_interp,
        'p_mlp':    p_mlp
    })

    print(
        f"{fold['name']} | "
        f"Interp={mean_squared_error(ground_truth, p_interp):.6f} | "
        f"MLP={mean_squared_error(ground_truth, p_mlp):.6f}"
    )

Fold 1 | Interp=0.000008 | MLP=0.000037
Fold 2 | Interp=0.000013 | MLP=0.000021


In [7]:
cascade_mses = []

for fold_idx, fold in enumerate(FOLDS):

    train_df = df[df['datetime'].dt.date.isin(fold['train'])].copy()
    val_df   = df[df['datetime'].dt.date.isin(fold['val'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)
    mask_rows = val_masked.loc[hidden_rows]

    train_obs = train_df[train_df['iv'].notna()]
    X_all     = train_obs[FEATURE_COLS].fillna(0).values
    y_all     = train_obs['iv'].values

    split = int(0.85 * len(X_all))
    model, scaler, best_ep, train_losses, val_losses = train_mlp(
        X_all[:split], y_all[:split],
        X_all[split:], y_all[split:]
    )

    preds = predict_cascaded_full(
        missing_rows=mask_rows,
        df_context=val_masked,
        mlp_model=model,
        mlp_scaler=scaler,
        expiry_date=EXPIRY_DATE,
    )

    mse = mean_squared_error(ground_truth, preds)
    cascade_mses.append(mse)

    print(f"{fold['name']} | Cascaded MSE = {mse:.6f}")

print("\nOverall Cascaded CV MSE:")
print(np.mean(cascade_mses))

Fold 1 | Cascaded MSE = 0.000046
Fold 2 | Cascaded MSE = 0.000028

Overall Cascaded CV MSE:
3.7209381518896013e-05
